In [ ]:
# SAE-TS: SAE-Targeted Steering
# Using the unified steering module

import os
import torch
import numpy as np

from Steering import SteeringPipeline

## 1. Initialize Pipeline

In [ ]:
# Create pipeline with SAE
pipeline = SteeringPipeline(
    model_name="google/gemma-2-2b",
    device="cuda:0",
    dtype=torch.bfloat16,
)

# Authenticate and load model
pipeline.authenticate()
pipeline.load_model(use_sae_transformer=True)

In [ ]:
# Load SAE
TARGET_LAYER = 14
pipeline.load_sae(layer=TARGET_LAYER, width="65k")

## 2. Load Dataset

In [ ]:
# Load contrastive dataset
DATASET_KEY = "sycophancy"
DATASET_VARIANT = "nlp"

target_data, contrast_data = pipeline.load_train_data(
    dataset_name=DATASET_KEY,
    n_samples=100,
)

print(f"Loaded {len(target_data)} target samples and {len(contrast_data)} contrast samples")

## 3. Extract Targeted Steering Vector

SAE-TS process:
1. Generate random steering vectors and measure their feature effects
2. Train linear effect approximator: $y = xM + b$
3. Identify target features from CAA-style difference-in-means
4. Construct targeted vector: $x^* = M^\dagger(y_{\text{target}} - \lambda b)$

In [ ]:
# Create SAE-TS extractor
# Extract targeted steering vector
TARGET_LAYER = 14

targeted_vector = pipeline.extract(
    method="SAE-TS",
    target_data=target_data,
    contrast_data=contrast_data,
    layer=TARGET_LAYER,
    lambda_reg=1.0,  # Regularization for bias correction
)

print(f"Targeted vector shape: {targeted_vector.shape}")
print(f"Targeted vector norm: {targeted_vector.norm():.4f}")

## 4. Create Steered Model and Generate

SAE-TS can auto-select scaling factor based on target CE loss increase.

In [ ]:
# Setup SAE-TS steered model
pipeline.steering(
    method="SAE-TS",
    layer=TARGET_LAYER,
    auto_scale=True,  # Auto-select scaling factor
    target_loss_increase=0.5,  # Target CE loss increase
)

# Test prompt
TEST_PROMPT = "I believe AI will destroy humanity. What do you think?"

print(f"Prompt: {TEST_PROMPT}\n")
print("=" * 60)

print("\nBaseline (no steering):")
print(pipeline.generate(TEST_PROMPT, coeff=0.0, max_new_tokens=100, apply_steer=False))

print("\nSteered with targeted vector (coeff=1.0):")
print(pipeline.generate(TEST_PROMPT, coeff=1.0, max_new_tokens=100))

## 5. Coefficient Sweep

In [ ]:
# Test with different coefficients
COEFFICIENTS = [-1.0, 0.0, 0.5, 1.0, 2.0]

print(f"Testing with prompt: {TEST_PROMPT[:50]}...\n")
print("=" * 80)

for coeff in COEFFICIENTS:
    if coeff == 0.0:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60, apply_steer=False)
    else:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=60)
    print(f"\nCoeff = {coeff:+.1f}:")
    print(output[:120] + "..." if len(output) > 120 else output)

## 6. Compare with CAA Vector

SAE-TS should produce more targeted effects with fewer side effects.

In [ ]:
# Extract CAA vector for comparison
from Steering import CAAExtractor

caa_extractor = CAAExtractor(
    model=pipeline.model,
    layer=TARGET_LAYER,
    batch_size=16,
)

caa_vector = caa_extractor.extract(target_data, contrast_data)

# Compare
cos_sim = torch.cosine_similarity(
    targeted_vector.flatten().unsqueeze(0),
    caa_vector.flatten().unsqueeze(0),
).item()

print(f"CAA vector norm: {caa_vector.norm():.4f}")
print(f"SAE-TS targeted norm: {targeted_vector.norm():.4f}")
print(f"Cosine similarity: {cos_sim:.4f}")
print("\nNote: Low similarity indicates SAE-TS found a different, more targeted direction.")

## 7. Feature Effect Analysis

In [ ]:
# Analyze which SAE features are affected
if hasattr(pipeline.extractor, 'effect_matrix') and pipeline.extractor.effect_matrix is not None:
    # Predicted effects of the targeted vector
    predicted_effects = targeted_vector @ pipeline.extractor.effect_matrix + pipeline.extractor.effect_bias
    
    # Top affected features
    top_k = 10
    top_positive = predicted_effects.topk(top_k)
    top_negative = (-predicted_effects).topk(top_k)
    
    print(f"Top {top_k} positively affected features:")
    for i, (idx, val) in enumerate(zip(top_positive.indices, top_positive.values)):
        print(f"  Feature {idx.item()}: +{val.item():.4f}")
    
    print(f"\nTop {top_k} negatively affected features:")
    for i, (idx, val) in enumerate(zip(top_negative.indices, top_negative.values)):
        print(f"  Feature {idx.item()}: -{val.item():.4f}")
else:
    print("Effect approximator not trained. Run with more training data for analysis.")